# GPU-Accelerated Quantum Simulation & Quantum Chemistry with NVIDIA CUDA-Q
### Statevector GHZ Benchmarking (CPU vs GPU) & H₂ VQE Potential Energy Surface Sweep

This Google Colab notebook runs the full GPU-accelerated suite using an **NVIDIA T4 / V100 / A100 GPU**.

**Hardware Setup:**
1. Go to **Runtime** > **Change runtime type**.
2. Select **T4 GPU** (or A100 if Colab Pro) under Hardware accelerator.
3. Click **Save**.

In [ ]:
# Step 1: Verify NVIDIA GPU Allocation
!nvidia-smi

In [ ]:
# Step 2: Install NVIDIA CUDA-Q & Quantum Chemistry Dependencies
!pip install --upgrade pip
!pip install cudaq pyscf openfermion openfermionpyscf matplotlib pandas scipy

In [ ]:
# Step 3: Inspect CUDA-Q Installation and Available Backends
import cudaq
print("NVIDIA CUDA-Q Version:", cudaq.__version__)
print("Available simulation targets:", cudaq.get_targets())

# Set target to GPU-accelerated cuStateVec
cudaq.set_target("nvidia")
print("Current active target set to:", "nvidia")

## Part 1: GHZ State Simulation Benchmark (CPU vs GPU)
Here we simulate Greenberger-Horne-Zeilinger (GHZ) circuits across qubit counts ($n = 20 \to 30$).
We compare the `qpp-cpu` target against the `nvidia` target (powered by `cuStateVec`).

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cudaq import h, x, mz

@cudaq.kernel
def ghz(n: int):
    q = cudaq.qvector(n)
    h(q[0])
    for i in range(n - 1):
        x.ctrl(q[i], q[i + 1])
    mz(q)

# Correctness validation at small n
cudaq.set_target("nvidia")
res = cudaq.sample(ghz, 8, shots_count=1000)
print("GHZ(8) Sample counts:", res)

# Benchmark loop
qubit_counts = list(range(20, 29))  # Up to 28 on T4 (16GB VRAM)
results = []

for target in ["qpp-cpu", "nvidia"]:
    cudaq.set_target(target)
    print(f"\nBenchmarking Target: {target}...")
    for n in qubit_counts:
        # 1 warm-up
        _ = cudaq.sample(ghz, n, shots_count=1000)
        # 5 timed runs
        times = []
        for _ in range(5):
            t0 = time.perf_counter()
            _ = cudaq.sample(ghz, n, shots_count=1000)
            times.append(time.perf_counter() - t0)
        med_s = np.median(times)
        std_s = np.std(times)
        mem_gb = (2**n * 8) / (1024**3)
        print(f"  n={n:2d} | Med: {med_s:8.5f}s (±{std_s:8.5f}s) | Mem: {mem_gb:.4f} GiB")
        results.append({"target": target, "n": n, "median_s": med_s, "std_s": std_s, "est_mem_gb": mem_gb})

df_ghz = pd.DataFrame(results)
df_ghz.to_csv("ghz_benchmark_colab.csv", index=False)

In [ ]:
# Plotting Runtime & Speedup
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=200)

# 1. Runtime vs n
for target, col in [("qpp-cpu", "#1f77b4"), ("nvidia", "#2ca02c")]:
    sub = df_ghz[df_ghz["target"] == target]
    ax1.plot(sub["n"], sub["median_s"], marker="o", label=f"{target.upper()} (fp32)", color=col, lw=2)

ax1.set_yscale("log")
ax1.set_xlabel("Number of Qubits ($n$)", fontweight="bold")
ax1.set_ylabel("Runtime (seconds) [Log Scale]", fontweight="bold")
ax1.set_title("CUDA-Q GHZ Statevector Simulation Runtime", fontweight="bold")
ax1.grid(True, which="both", ls="--", alpha=0.5)
ax1.legend()

# 2. Speedup (CPU / GPU)
cpu_sub = df_ghz[df_ghz["target"] == "qpp-cpu"].set_index("n")["median_s"]
gpu_sub = df_ghz[df_ghz["target"] == "nvidia"].set_index("n")["median_s"]
common_n = cpu_sub.index.intersection(gpu_sub.index)
speedup = cpu_sub.loc[common_n] / gpu_sub.loc[common_n]

ax2.plot(common_n, speedup, marker="s", color="#d62728", lw=2.5, label="cuStateVec Speedup")
ax2.axhline(1.0, color="gray", ls=":", label="Parity (1x)")
ax2.set_xlabel("Number of Qubits ($n$)", fontweight="bold")
ax2.set_ylabel("Speedup Factor ($T_{CPU} / T_{GPU}$)", fontweight="bold")
ax2.set_title("GPU Speedup over CPU (NVIDIA T4)", fontweight="bold")
ax2.grid(True, ls="--", alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

## Part 2: Molecular Hydrogen (H₂) VQE Potential Energy Surface Sweep
Construct Jordan-Wigner 15-term Hamiltonian, optimize UCCSD ansatz with `cudaq.observe`, and compare against exact PySCF Full Configuration Interaction (FCI).

In [ ]:
from pyscf import gto, scf, ao2mo, fci
from openfermion import InteractionOperator, get_fermion_operator, jordan_wigner
from openfermion.chem.molecular_data import spinorb_from_spatial
from scipy.optimize import minimize
from cudaq import spin, cx, h, ry, x

@cudaq.kernel
def h2_uccsd(theta: float):
    q = cudaq.qvector(4)
    x(q[0])
    ry(2.0 * theta, q[0])
    x(q[2])
    cx(q[0], q[2])
    cx(q[0], q[1])
    cx(q[2], q[3])

def build_hamiltonian(r: float):
    mol = gto.M(atom=f"H 0 0 0; H 0 0 {r}", basis="sto-3g", verbose=0)
    mf = scf.RHF(mol).run()
    n_orb = mf.mo_coeff.shape[1]
    h1e = mf.mo_coeff.T @ mf.get_hcore() @ mf.mo_coeff
    two_e = ao2mo.restore(1, ao2mo.kernel(mol, mf.mo_coeff), n_orb)
    h2e = np.asarray(two_e.transpose(0, 2, 3, 1), order="C")
    
    one_s, two_s = spinorb_from_spatial(h1e, h2e)
    ham = InteractionOperator(float(mol.energy_nuc()), one_s, 0.5 * two_s)
    q_ham = jordan_wigner(get_fermion_operator(ham))
    
    spin_h = 0.0 * spin.i(0)
    for term, coeff in q_ham.terms.items():
        c = float(np.real(coeff))
        if abs(c) < 1e-12: continue
        if len(term) == 0:
            spin_h = spin_h + c * spin.i(0)
        else:
            t_op = None
            for q_idx, p in term:
                op = spin.x(q_idx) if p == 'X' else spin.y(q_idx) if p == 'Y' else spin.z(q_idx)
                t_op = op if t_op is None else t_op * op
            spin_h = spin_h + c * t_op
    
    e_fci, _ = fci.FCI(mf).kernel()
    return spin_h, float(mf.e_tot), float(e_fci)

# Sweep PES from 0.30 to 2.50 Angstroms
r_grid = np.arange(0.30, 2.55, 0.05)
pes_data = []
warm_th = 0.1

for r in r_grid:
    h_op, e_hf, e_fci = build_hamiltonian(r)
    res = minimize(lambda p: cudaq.observe(h2_uccsd, h_op, float(p[0])).expectation(), [warm_th], method="COBYLA")
    warm_th = float(res.x[0])
    e_vqe = float(res.fun)
    err_mha = abs(e_vqe - e_fci) * 1000.0
    pes_data.append({"R": r, "E_vqe": e_vqe, "E_fci": e_fci, "E_hf": e_hf, "abs_error_mHa": err_mha})

df_pes = pd.DataFrame(pes_data)
print("PES Sweep completed!")
print(f"Maximum error across sweep: {df_pes['abs_error_mHa'].max():.6f} mHa (Chemical Accuracy < 1.60 mHa)")

In [ ]:
# Plot Potential Energy Surface & Error Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), dpi=200)

# 1. PES Curve
ax1.plot(df_pes["R"], df_pes["E_fci"], label="FCI Exact (PySCF)", color="#1f77b4", lw=2.5)
ax1.plot(df_pes["R"], df_pes["E_hf"], label="Hartree-Fock (RHF)", color="#d62728", ls="--", lw=1.8)
ax1.scatter(df_pes["R"], df_pes["E_vqe"], label="CUDA-Q VQE (UCCSD)", color="#ff7f0e", s=40, zorder=5)
ax1.axhline(-0.933164, color="gray", ls=":", label="2 × E(H atom) asymptote")
ax1.set_xlabel("Interatomic Distance $R$ (Å)", fontweight="bold")
ax1.set_ylabel("Total Energy (Hartree)", fontweight="bold")
ax1.set_title("H₂ Potential Energy Surface", fontweight="bold")
ax1.set_ylim(-1.20, -0.70)
ax1.grid(True, ls="--", alpha=0.5)
ax1.legend()

# 2. Absolute Error vs FCI
ax2.plot(df_pes["R"], df_pes["abs_error_mHa"], marker="o", color="#1f77b4", lw=2)
ax2.axhspan(0, 1.60, color="#2ca02c", alpha=0.2, label="Chemical Accuracy (< 1.60 mHa)")
ax2.axhline(1.60, color="#2ca02c", ls="--", lw=1.5)
ax2.set_xlabel("Interatomic Distance $R$ (Å)", fontweight="bold")
ax2.set_ylabel("|E_VQE - E_FCI| (mHa)", fontweight="bold")
ax2.set_title("VQE Energy Error relative to FCI", fontweight="bold")
ax2.set_ylim(0, 2.0)
ax2.grid(True, ls="--", alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()